# set enviromental variables

Make sure you set these two enviromental variables:

* `HOOD_PROJECT` should lead to the HOOD repository
* `HOOD_DATA` should lead to the data folder (see `README.md` for details)

In [1]:
import os

os.environ["HOOD_PROJECT"] ="/home/zhudf/projects/HOOD/"
os.environ["HOOD_DATA"] =  "/home/zhudf/data/hood_data/hood_data"

In [2]:
HOOD_PROJECT = os.environ["HOOD_PROJECT"]
HOOD_DATA = os.environ["HOOD_DATA"]

# Prepare pose sequences

To infer the HOOD models over a specific sequence of body poses, you first need to convert the pose sequence into a `.pkl` file:

The `.pkl` file should contain a dictionary with the following contents
* `body_pose`: numpy array of shape \[Nx69\] with a sequence of SMPL pose parameters
* `global_orient`: global orientations of the body, \[Nx3\] array
* `transl`: global translations of the body, \[Nx3\] array
* `betas`: SMPL shape parameters, \[10,\] array


Here we provide the functions to generate such files from two sources:
* VTO dataset (clone [this repository](https://github.com/isantesteban/vto-dataset) to download, all data is in the repo)
* AMASS dataset (download **CMU** split with **SMPL+H** parameters from [here](https://amass.is.tue.mpg.de/))

要在特定的身体姿态序列上对 HOOD 模型进行推断，首先需要将姿态序列转换为 .pkl 文件：
该 .pkl 文件应包含一个字典，其内容如下：
• body_pose：形状为 [Nx69] 的 numpy 数组，包含一组 SMPL 姿态参数  其中N的意思是帧的数量
• global_orient：身体的全局方向，[Nx3] 数组
• transl：身体的全球平移，[Nx3] 数组
• betas：SMPL 形状参数，[10,] 数组
此处我们提供了从以下两个来源生成此类文件的函数：
• VTO 数据集（克隆此仓库以下载，所有数据均在仓库中）
• AMASS 数据集（从这里下载包含 SMPL+H 参数的 CMU 分割数据）

## sequence from the VTO dataset &rarr;  HOOD .pkl

In [3]:
#from pathlib import Path
#from utils.data_making import convert_vto_to_pkl


#VTO_DATASET_PATH = '/path/to/vto-dataset/'

#vto_sequence_path = Path(VTO_DATASET_PATH) / 'tshirt/simulations/tshirt_shape00_01_01.pkl'
#target_pkl_path =  Path(HOOD_DATA) / 'temp/01_01.pkl'


#convert_vto_to_pkl(vto_sequence_path, target_pkl_path, n_zeropose_interpolation_steps=30)
#print(f'Pose sequence saved into {target_pkl_path}')

FileNotFoundError: [Errno 2] No such file or directory: '/path/to/vto-dataset/tshirt/simulations/tshirt_shape00_01_01.pkl'

## AMASS .npz sequences &rarr;  HOOD .pkl

In [ ]:
from utils.data_making import convert_amass_to_pkl
from pathlib import Path
AMASS_DATASET_PATH = '/home/zhudf/data'

amass_seq_path = Path(AMASS_DATASET_PATH) / 'CMU/01/01_01_poses.npz'
target_pkl_path =  Path(HOOD_DATA) / 'temp/01_01.pkl'

convert_amass_to_pkl(amass_seq_path, target_pkl_path, target_fps=30)
print(f'Pose sequence saved into {target_pkl_path}')

# Choose a garment

Next, you need to choose a garment to simulate.

Its template and some auxiliary data should be stored in the `$HOOD_DATA/aux_data/garments_dict.pkl` file

You can choose from the list of garments already provided in this file:

![all_garments](static/all_garments.png)

Or you can import a new garment from an `.obj` file

We also provide `.obj` files for all garments usen in the paper in `$HOOD_DATA/aux_data/garment_meshes/` directory.
Note that these `.obj` files only have demonstrational purpose. 
For inference and training we use garment data stored in  `$HOOD_DATA/aux_data/garments_dict.pkl`.

In [ ]:
选择一件服装

接下来，您需要选择一件要进行模拟的服装。
该服装的模板及部分辅助数据应存储在 $HOOD_DATA/aux_data/garments_dict.pkl 文件中。
您可以从该文件中已提供的服装列表中进行选择：
或者，您可以从 .obj 文件导入一件新服装
我们还在 $HOOD_DATA/aux_data/garment_meshes/ 目录中提供了论文中使用的一切服装的 .obj 文件。请注意，这些 .obj 文件仅用于演示目的。在推理和训练过程中，我们使用存储在 $HOOD_DATA/aux_data/garments_dict.pkl 中的服装数据。

## Add your own garment from an `.obj` file

First, add the garment to the `garments_dict.pkl` file using `add_garment_to_garments_dict` function.

It builds a dictionary for the garment that contains:
* `rest_pos`: \[Nx3\], positions of the vertices in canonical pose that are aligned to zero- pose and shape SMPL body.
* `faces`: \[Fx3\], triplets of node indices that constitute each face
* `node_type` \[Nx1\], node type labels (`0` for regular, `3` for "pinned"). By default, all nodes are regular, we show how to add "pinned nodes" later in this notebook
* `lbs` dictionary with shape- and pose- blendshapes and skinning weights for the garment, sampled from SMPL model
* `center` and `coarse_edges` info on long-range (coarse) edges used to build hiererchical graph of the garment.

To be able to start simulation from arbitrary pose, we use linear blend-skinning (LBS) to initialize the garment geometry in the first frame. For each garment node we sample pose-and shape-blendshapes and skinning weights from the closest SMPL node in canonical pose.

However, for loose garments such approach may result in overly-stretched triangles. Therefore, we use the approach introduced in [\[Santesteban et al. 2021\]](http://mslab.es/projects/SelfSupervisedGarmentCollisions/) and average skinning weights and blendshapes over many randomly sampled SMPL nodes around the given garment node.

The parameter `n_samples_lbs` controls the number of random samples to use. We recommend setting it to 0 for tight-fitting garments (shirts, pants) and to 1000 for loose ones (skirts, dresses).

从 .obj 文件中添加您自己的服装

首先，使用 add_garment_to_garments_dict 函数将服装添加到 garments_dict.pkl 文件中。
该函数会为服装构建一个字典，其中包含：
• rest_pos: [Nx3]，标准姿势下与零姿势和SMPL身体形状对齐的顶点位置。
• faces: [Fx3]，构成每个面的节点索引三元组
• node_type [Nx1]，节点类型标签（0 表示常规节点，3 表示“固定节点”）。默认情况下，所有节点均为常规节点，我们将在本笔记本的后续部分展示如何添加“固定节点”
• lbs 字典，其中包含服装的形状和姿态混合形状以及蒙皮权重，这些数据采样自 SMPL 模型
• 中心点和粗边信息，用于构建服装的分层图。
为了能够从任意姿态开始模拟，我们使用线性混合蒙皮（LBS）在第一帧初始化服装几何结构。对于每个服装节点，我们从处于规范姿态的最接近的 SMPL 节点中采样姿态和形状混合形状以及蒙皮权重。
然而，对于宽松的服装，这种方法可能会导致三角形过度拉伸。因此，我们采用 [Santesteban 等，2021] 中提出的方法，对给定服装节点周围随机采样的多个 SMPL 节点，对其蒙皮权重和混合形状进行求平均。
参数 n_samples_lbs 控制要使用的随机采样数量。我们建议对于紧身服装（衬衫、裤子）将其设置为 0，对于宽松服装（裙子、连衣裙）则设置为 1000。

garments_dict.pkl
│
├── tshirt
│   ├── rest_pos
│   ├── faces
│   ├── node_type
│   ├── lbs
│   ├── center
│   └── coarse_edges
│
├── pants
│   ├── rest_pos
│   ├── faces
│   ├── node_type
│   ├── lbs
│   ├── center
│   └── coarse_edges
│
└── longskirt
    ├── rest_pos
    ├── faces
    ├── node_type
    ├── lbs
    ├── center
    └── coarse_edges

    
garment_dict = {

    "rest_pos": ...,

    "faces": ...,

    "node_type": ...,

    "lbs": {
        "v": ...,
        "f": ...,
        "shapedirs": ...,
        "posedirs": ...,
        "lbs_weights": ...
    },

    "center": ...,

    "coarse_edges": ...
}

garment_dict = make_garment_dict(...)
garments_dict[garment_name] = garment_dict

graph center：在服装网格图里，选一个“拓扑上最居中”的顶点，作为分层粗化的参考点。
coarse_edges：在原始 mesh edge 之外，再人为构造一些“更长距离”的边，让 GNN 可以更快地把信息从衣服的一处传到远处。

In [ ]:
import os
from utils.mesh_creation import add_garment_to_garments_dict, add_pinned_verts
from utils.defaults import DEFAULTS

garment_obj_path = os.path.join(DEFAULTS.aux_data, 'garment_meshes','hood', 'longskirt.obj')
smpl_file = os.path.join(DEFAULTS.aux_data, 'smpl', 'SMPL_FEMALE.pkl')
garments_dict_path = os.path.join(DEFAULTS.aux_data, 'garments_dict.pkl')

# Name of the garment we are adding
garment_name = 'longskirt'

add_garment_to_garments_dict(garment_obj_path, garments_dict_path, garment_name, smpl_file=smpl_file, n_samples_lbs=1000, verbose=True)


### Add pinned vertices

For some gaments, it can be necessary to fix positions of a subset of nodes relative to the body. For example, fix the top ring of a skirt or pants to prevent it from falling off the body.

To label a set of garment nodes as "pinned", you need to use `add_pinned_verts` function and provide it with the list of node indices that you want to pin.

One easy way of getting indices for a set of nodes, is by using [Blender](https://www.blender.org/). 

1. Open it, import the garment from the `.obj` file. 
2. Then in `Layout` tab press `Tab` to go into the editing mode. 
3. Select all vertices you want to pin. 
4. Then, go into `Scripting` tab and execute the following piece of code there.

```python
import bpy
import bmesh

obj = bpy.context.active_object
bm = bmesh.from_edit_mesh(obj.data)    
obj = bpy.context.active_object; bm = bmesh.from_edit_mesh(obj.data)    ; selected = [i.index for i in bm.verts if i.select == True]; print(selected)

```


5. You will get a list of indices for the selected nodes.

添加固定顶点¶
对于某些服装，可能需要将一部分节点的相对位置固定在身体上。例如，将裙子或裤子的顶部环形部分固定住，以防止其从身体上滑落。
要将一组服装节点标记为“固定”，需要使用 add_pinned_verts 函数，并向其提供要固定的节点索引列表。
获取一组顶点索引的简便方法之一是使用 Blender。
1. 打开 Blender，从 .obj 文件中导入服装模型。
2. 然后在“布局”选项卡中按 Tab 键进入编辑模式。
3. 选中所有需要固定的顶点。
4. 接着，进入“脚本”选项卡，并在其中执行以下代码片段。

In [ ]:
pinned_indices = \
[2196, 2197, 2198, 2199, 2200, 2201, 2202, 2203, 2204, 2205, 2206, 2207, 2208, 2209, 2210, 2211, 2212, 2213, 2214, 2215,
 2216, 2217, 2218, 2219, 2220, 2221, 2222, 2223, 2224, 2225, 2226, 2227, 2228, 2229, 2230, 2231, 2232, 2233, 2234, 2235,
 2236, 2237, 2238, 2239, 2240, 2241, 2242, 2243, 2244, 2327, 2328, 2351, 2492, 2497, 2669, 2670, 2671, 2674, 2837, 3139,
 3200, 3204, 3359, 3362, 3495, 3512, 3634, 3638, 3805, 3965, 3967, 4133, 4137, 4140, 4335, 4340, 4506, 4509, 4669, 4674,
 4749, 4812, 4849, 4853, 5138, 5309, 5342, 5469, 5474, 5503, 5646, 5650, 5654, 5855, 5857, 6028, 6091, 6204, 6209, 6280,
 6374, 6377, 6378, 6473, 6649, 6654, 6814, 6817, 6986, 6989, 6990, 6992, 7172, 7178, 7336, 7500, 7505, 7661, 7665, 7666]

add_pinned_verts(garments_dict_path, garment_name, pinned_indices)

# Generate rollout for one sequence

Once we created a `.pkl` file wth a pose sequence and added our garment to the `garments_dict.pkl` (or you can use one of the garments that are already there), we can generate a rollout sequence for them using a trained HOOD model.

We provide 4 pretrained models and corresponding configuration files for each of them. The weights of the trained models are located in `$HOOD_DATA/trained_models`. The configuration files are in  `$HOOD_PROJECT/configs`

| model file      | config name           | comments                                                                                                                                                                                                                            |
|-----------------|-----------------------|-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| fine15          | cvpr_baselines/fine15 | Baseline model denoted as `Fine15` in the paper. No long-range edges, 15 message-passing steps.                                                                                                                                     |
| fine18          | cvpr_baselines/fine48 | Baseline model denoted as `Fine48` in the paper. No long-range edges, 48 message-passing steps.                                                                                                                                     |
| cvpr_submission | cvpr                  | Model used in the CVPR paper. Use it if you want to compare to the paper.                                                                                                                                                           |
| postcvpr        | postcvpr              | Newer model trained using refactored code with minor bug fixes. Use it if you want to use HOOD model in a downstream task.  Also use this config if you want to train a HOOD model from scratch (see `Training.ipynb` for details) |

为一个序列生成展开序列

一旦我们创建了一个包含姿态序列的 .pkl 文件，并将我们的服装添加到 garments_dict.pkl 中（或者您可以使用其中现有的服装之一），就可以使用经过训练的 HOOD 模型为它们生成展开序列。
我们提供了 4 个预训练模型，并为每个模型提供了相应的配置文件。训练模型的权重位于 $HOOD_DATA/trained_models 目录下。配置文件位于 $HOOD_PROJECT/configs 目录下。

### create validation config and create `Runner` object

In [ ]:
validation config
      ↓
告诉程序：
- 用哪个模型
- 用哪个 checkpoint
- 在哪块 GPU 上跑
- 用什么数据
- rollout 怎么设置
- 是否保存结果
- 其他验证参数
      ↓
Runner
      ↓
把这些配置真正变成可执行流程
      ↓
runner.xxx(...)
      ↓
加载数据
调用模型
逐帧 rollout
输出结果

In [ ]:
训练时可能需要：
optimizer
learning_rate
backward
epochs
gradient

而 validation/inference 更关注：
加载训练好的模型
加载 checkpoint
准备输入
forward
rollout
保存预测结果

In [3]:
from utils.validation import Config as ValidationConfig
from utils.validation import load_runner_from_checkpoint, update_config_for_validation, create_one_sequence_dataloader
from utils.arguments import load_params
from utils.common import move2device, pickle_dump
from utils.defaults import DEFAULTS
from pathlib import Path


# Set material paramenters, see configs/cvpr.yaml for the training ranges for each parameter
config_dict = dict()
config_dict['density'] = 0.20022
config_dict['lame_mu'] = 23600.0
config_dict['lame_lambda'] = 44400
config_dict['bending_coeff'] = 3.962e-05

# If True, the SMPL poses are slightly modified to avoid hand-body self-penetrations. The technique is adopted from the code of SNUG 
config_dict['separate_arms'] = True

# Paths to SMPL model and garments_dict file relative to $HOOD_DATA/aux_data
config_dict['garment_dict_file'] = 'garments_dict.pkl'
config_dict['smpl_model'] = 'smpl/SMPL_FEMALE.pkl'

validation_config = ValidationConfig(**config_dict)


# Choose the model and the configuration file

# config_name = 'cvpr_baselines/fine15'  
# checkpoint_path = Path(DEFAULTS.data_root) / 'trained_models' / 'fine15.pth'

# config_name = 'cvpr_baselines/fine48'
# checkpoint_path = Path(DEFAULTS.data_root) / 'trained_models' / 'fine48.pth'

# config_name = 'cvpr'
# checkpoint_path = Path(DEFAULTS.data_root) / 'trained_models' / 'cvpr_submission.pth'

config_name = 'postcvpr'
checkpoint_path = Path(DEFAULTS.data_root) / 'trained_models' / 'postcvpr.pth'

# load the config from .yaml file and load .py modules specified there
modules, experiment_config = load_params(config_name)

# modify the config to use it in validation 
experiment_config = update_config_for_validation(experiment_config, validation_config)

# load Runner object and the .py module it is declared in
runner_module, runner = load_runner_from_checkpoint(checkpoint_path, modules, experiment_config)


### create one-sequence dataloader

In [ ]:
原始数据可能有：
body_pose
global_orient
transl
betas
garment_name

而 HOOD 真正运行时还可能需要把它整理成图数据，比如：
服装节点
mesh edges
coarse edges
人体节点
节点特征
顶点位置
速度
node_type

所以 dataloader 的作用就是：
把原始输入整理、转换、打包成模型真正需要的输入。

In [4]:
# file with the pose sequence
sequence_path =  Path(HOOD_DATA) / 'temp/01_01.pkl'


# name of the garment to sumulate
garment_name = 'longskirt'

dataloader = create_one_sequence_dataloader(sequence_path, garment_name, modules, experiment_config)

In [5]:
sequence = next(iter(dataloader))
sequence = move2device(sequence, 'cuda:0')
trajectories_dict = runner.valid_rollout(sequence,  bare=True)

100%|█████████████████████████████████████████████████████████████| 686/686 [00:33<00:00, 20.58it/s]


In [6]:
# Save the sequence to disc
out_path = Path(DEFAULTS.data_root) / 'temp' / 'output.pkl'
print(f"Rollout saved into {out_path}")
pickle_dump(dict(trajectories_dict), out_path)

Rollout saved into /home/zhudf/data/hood_data/hood_data/temp/output.pkl


In [7]:
from utils.common import pickle_load
trajectories_dict['pred'].shape

sequence_path =  Path(HOOD_DATA) / 'temp/01_01.pkl'

sequence = pickle_load(sequence_path)

In [8]:
out_path

PosixPath('/home/zhudf/data/hood_data/hood_data/temp/output.pkl')

### write video

Finally, we can render a video of the generated sequence with [aitviewer](https://github.com/eth-ait/aitviewer)

Or you can render it interactively using `python utils/show.py rollout_path=PATH_TO_SEQUENCE`

In [3]:



from utils.show import write_video
from aitviewer.headless import HeadlessRenderer

renderer = HeadlessRenderer()

In [4]:
print(renderer.ctx.info["GL_VENDOR"])
print(renderer.ctx.info["GL_RENDERER"])
print(renderer.ctx.info["GL_VERSION"])

NVIDIA Corporation
NVIDIA GeForce RTX 3090 Ti/PCIe/SSE2
4.5.0 NVIDIA 570.133.07


In [5]:
import skvideo

skvideo.setFFmpegPath("/home/zhudf/miniconda3/envs/hood/bin")


import skvideo.io
print(skvideo.io.ffmpeg._HAS_FFMPEG)

1


In [6]:
from pathlib import Path
from utils.defaults import DEFAULTS
out_path = Path(DEFAULTS.data_root) / 'temp' / 'output_n.pkl'
out_video = Path(DEFAULTS.data_root) / 'temp' / 'output_n.mp4'
write_video(out_path, out_video, renderer)

Rendering frames: 100%|████████████████████████████████████████████| 772/772 [00:12<00:00, 62.22it/s]

Video saved to /home/zhudf/data/hood_data/hood_data/temp/output_n.mp4
